In [1]:
import os
import warnings
import importlib

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, GroupKFold, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from itertools import combinations
import sys
sys.path.append('/kaggle/input/datasets/keithmarange/hist-boost-methods/')
sys.path.append('/kaggle/input/cmi-competition-code')
import utils
from scipy.stats import uniform, randint

warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', category=RuntimeWarning)

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import ParameterSampler, RandomizedSearchCV

from scipy.stats import uniform, randint, loguniform

In [2]:
data_folder = utils.find_data_root()

raw_train_df  = pd.read_csv(data_folder / 'train.csv')
raw_test_df   = pd.read_csv(data_folder / 'test.csv')
train_demo_df = pd.read_csv(data_folder / 'train_demographics.csv')
test_demo_df  = pd.read_csv(data_folder / 'test_demographics.csv')

temp_calculations_folder_name = 'temp_calculations/'
model_run_folder_name = 'model_runs/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)
os.makedirs(model_run_folder_name, exist_ok=True)

Using Kaggle data folder: /kaggle/input/competitions/cmi-detect-behavior-with-sensor-data


In [3]:
model_run_name = 'hist_boost_v1'

feat_columns = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture']
acc_columns  = ['acc_x', 'acc_y', 'acc_z']
rot_columns  = ['rot_w','rot_x', 'rot_y', 'rot_z']
thm_columns = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']

non_device_cols = [
    'sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation',
    'behavior', 'phase', 'gesture', 'adult_child', 'age', 'sex', 'handedness',
    'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm'
]

sampling_rate = 100
dc_offset_max = 2
pipe_name = 'imu_extractor'

linear_edges = np.arange(0, 51, 10)
log_edges    = np.logspace(np.log10(0.5), np.log10(50), num=10)
custom_edges = np.array([0, 1, 2, 4, 8, 15, 25, 50])

n_splits = 3
tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(0, 64)]

tof_1 = [f'tof_1_v{j}' for j in range(64)]
tof_2 = [f'tof_2_v{j}' for j in range(64)]
tof_3 = [f'tof_3_v{j}' for j in range(64)]
tof_4 = [f'tof_4_v{j}' for j in range(64)]
tof_5 = [f'tof_5_v{j}' for j in range(64)]

some_sequences = ['SEQ_000007', 'SEQ_000008', 'SEQ_000013', 'SEQ_000034', 'SEQ_000046', 'SEQ_000053']

thm_columns = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']

orientation_cols = [
    'Seated Straight',
    'Lie on Side - Non Dominant',
    'Seated Lean Non Dom - FACE DOWN',
    'Lie on Back'
]

orientation_cols_dict = {
    'Seated Straight': ['Seated Straight'],
    'Lie on Side': ['Lie on Side - Non Dominant'],
    'Seated Lean': ['Seated Lean Non Dom - FACE DOWN'],
    'Lie on Back': ['Lie on Back'],
    'All': orientation_cols
}

gesture_position_cols_dict = {
    'All': ['Above ear', 'Cheek', 'Eyebrow', 'Eyelash', 'Forehead', 'Neck'],
    'Forehead': ['Forehead'],
    'Neck': ['Neck'],
    'Eyebrow': ['Eyebrow'],
    'Eyelash': ['Eyelash'],
    'Cheek': ['Cheek'],
    'Above ear': ['Above ear'],
     'Forehead and Neck': ['Forehead','Neck'],
}

model_target_list = ['gesture_position', 'gesture_action', 'gesture']

do_report = False
save_model = False
random_search = True

orientation_cols_dict = {
    'Lie on Back': ['Lie on Back'],
}

model_target_list = ['gesture_action']

In [4]:
train_df = raw_train_df.set_index('row_id')
target_only_df = train_df[train_df['sequence_type'] == 'Target'].copy()

target_only_df['gesture_position'] = target_only_df['gesture'].str.split(' - ').str[0]
target_only_df['gesture_action'] = target_only_df['gesture'].str.split(' - ').str[-1]

train_sample_df, test_sample_df = utils.sample_balanced_split(
    target_only_df,
    train_pct=0.2,
    test_pct=0.2
)

# some_sequences = train_sample_df['sequence_id'].unique()[:50]
# train_sample_df = train_sample_df[train_sample_df['sequence_id'].isin(some_sequences)]

Train: 648 seqs | 12.7%
Test:  648 seqs  | 12.7%


In [5]:
importlib.reload(utils)

num_pattern  = 'acc|rot|thm|tof'
suspect_cols = ['age', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']
cat_cols     = ['orientation']
normal_cols  = ['adult_child', 'sex', 'handedness']
ordinal_cols = ['segment_id']

preprocessor = ColumnTransformer(
    transformers=[
        (
            'feature_num_cols',
            StandardScaler(),
            make_column_selector(pattern=num_pattern)
        ),
        (
            'subject_num_cols',
            StandardScaler(),
            utils.existing_cols(suspect_cols)
        ),
        (
            'cat_encoder',
            OneHotEncoder(handle_unknown='ignore', sparse_output=False),
            utils.existing_cols(cat_cols)
        ),
        (
            'normal_cols',
            'passthrough',
            utils.existing_cols(normal_cols)
        ),
        (
            'ordinal_cols',
            'passthrough',
            utils.existing_cols(ordinal_cols)
        )
    ],
    remainder='drop',
    verbose_feature_names_out=False
)
preprocessor.set_output(transform='pandas')

param_grid = {
    f'{pipe_name}__imu_sensor_list':        [acc_columns],
    f'{pipe_name}__imu_domain':             ['velocity'],
    f'{pipe_name}__combine_imu_axes':       [True],
    f'{pipe_name}__sampling_rate':          [100],

    f'{pipe_name}__rotation_sensor_list':   [rot_columns],
    f'{pipe_name}__combine_rot_axes':       [False],
    f'{pipe_name}__rotation_domain':        ['orientation'],

    f'{pipe_name}__thermopile_sensor_list': [thm_columns],
    f'{pipe_name}__thermopile_mode':        ['spatial'],
    f'{pipe_name}__tof_sensor_list':        [tof_columns],
    f'{pipe_name}__tof_mode':               ['blob'],

    f'{pipe_name}__dc_offset':              [0],
    f'{pipe_name}__band_edges':             [log_edges],
    f'{pipe_name}__category_data':          [False],
    f'{pipe_name}__segmentation':           ['window'],

    'pca__n_components': [None], 

    'classifier__estimator__learning_rate': loguniform(0.005, 0.1),  # Log scale for learning rate
    'classifier__estimator__max_iter': randint(50, 300),  # Integer between 50-300
    'classifier__estimator__max_depth': randint(3, 8),  # Integer between 3-8
    'classifier__estimator__min_samples_leaf': randint(10, 50),  # Integer between 10-50
    'classifier__estimator__l2_regularization': loguniform(0.1, 10.0),  # Log scale
    'classifier__estimator__max_leaf_nodes': randint(10, 50),  # Integer between 10-50
    'classifier__estimator__early_stopping': [True],  # Fixed value
    'classifier__estimator__n_iter_no_change': randint(5, 20),  # Integer between 5-20
    'classifier__estimator__tol': loguniform(1e-5, 1e-3),  # Log scale for tolerance
}

param_grid[f'{pipe_name}__window'] = [1.2]
param_grid[f'{pipe_name}__step_sec'] = [0.2]

custom_extractor = utils.ImuExtractor(subject_df=train_demo_df)

# HistGradientBoosting with anti-overfitting defaults
hgb_clf = HistGradientBoostingClassifier(
    loss='log_loss',
    random_state=42,
    verbose=0
)

In [6]:
for model_target in model_target_list:

    cv_results_list = []
    
    for col in orientation_cols_dict:
        pipeline = Pipeline([
                    (pipe_name, custom_extractor),
                    ('preprocessor', preprocessor),
                    ('pca', utils.IndexPreservingPCA()),
                    ('classifier', utils.ManyToOneWrapper(
                        estimator=hgb_clf,
                        extractor=custom_extractor,
                        mode=None,
                        target=model_target))])

        if random_search:
            search_obj = RandomizedSearchCV(
                            estimator=pipeline,
                            param_distributions=param_grid,
                            n_iter=15,                    # 👈 Only 5 combinations
                            cv=GroupKFold(n_splits=n_splits),                        # 3-fold cross-validation
                            random_state=42,             # Reproducible randomness
                            n_jobs=-1,                   # Use all cores
                            verbose=1,                   # See progress
                            return_train_score=True      # Track overfitting
                        )
        else:
            search_obj = GridSearchCV(
                                estimator=pipeline, 
                                param_grid=param_grid,  
                                cv=GroupKFold(n_splits=n_splits),
                                verbose=1, 
                                n_jobs=-1, 
                                return_train_score=True
                        )
        
        sliced_data_df = train_sample_df[train_sample_df['orientation'].isin(orientation_cols_dict[col])]
        y = sliced_data_df[['sequence_id', model_target]]
        search_obj.fit(sliced_data_df, y, groups=sliced_data_df['sequence_id'])

        if save_model:
            model = search_obj.best_estimator_
            path_to_model_run_name = model_run_folder_name + f'{model_run_name}_{col}_{model_target}.pkl'
            joblib.dump(model, path_to_model_run_name)

        cv_results_df = pd.DataFrame(search_obj.cv_results_)
        cv_results_df['orientation_data'] = col
        cv_results_list.append(cv_results_df)
        
    master_cv_results_df = pd.concat(cv_results_list)
    master_cv_results_df['model_target'] = model_target
    path_to_cv_results = model_run_folder_name + f'{model_run_name}_{model_target}_results.csv'
    master_cv_results_df.to_csv(path_to_cv_results, index=False)

Fitting 3 folds for each of 15 candidates, totalling 45 fits


/kaggle/input/datasets/keithmarange/hist-boost-methods/utils.py:1005: RuntimeWarning: Mean of empty slice
  mean_frame = np.nanmean(frames, axis=0)
/kaggle/input/datasets/keithmarange/hist-boost-methods/utils.py:1005: RuntimeWarning: Mean of empty slice
  mean_frame = np.nanmean(frames, axis=0)
/kaggle/input/datasets/keithmarange/hist-boost-methods/utils.py:1005: RuntimeWarning: Mean of empty slice
  mean_frame = np.nanmean(frames, axis=0)
/kaggle/input/datasets/keithmarange/hist-boost-methods/utils.py:1005: RuntimeWarning: Mean of empty slice
  mean_frame = np.nanmean(frames, axis=0)
/kaggle/input/datasets/keithmarange/hist-boost-methods/utils.py:1005: RuntimeWarning: Mean of empty slice
  mean_frame = np.nanmean(frames, axis=0)
/kaggle/input/datasets/keithmarange/hist-boost-methods/utils.py:1005: RuntimeWarning: Mean of empty slice
  mean_frame = np.nanmean(frames, axis=0)
/kaggle/input/datasets/keithmarange/hist-boost-methods/utils.py:1005: RuntimeWarning: Mean of empty slice
  mean

In [7]:
if do_report:
    
    best_model = search_obj.best_estimator_

    extractor    = best_model.named_steps['imu_extractor']
    preprocessor = best_model.named_steps['preprocessor']
    classifier   = best_model.named_steps['classifier']

    X_feat = extractor.transform(test_sample_df)
    X_proc = preprocessor.transform(X_feat)

    y_true = test_sample_df.drop_duplicates('sequence_id').set_index('sequence_id')['gesture']
    y_true = y_true.reindex(X_proc.index)

    y_pred = pd.Series(classifier.predict(X_proc), index=X_proc.index)

    print(classification_report(y_true, y_pred))

    report_df = pd.DataFrame(
        classification_report(y_true, y_pred, output_dict=True)
    ).T.sort_values('f1-score', ascending=False)

    report_df.to_csv(model_run_folder_name + 'svm_rbf_per_gesture_scores.csv')